# Manual_Example — 手册主算例: 光阴极枪 + 螺线管, 发射度补偿

本 notebook 是 06 汇总 (notebooks/06_examples.ipynb) 的细分教学: 只跑这一个官方算例, 逐步解读输入卡、展示本算例最有代表性的图。

In [ ]:
%run ../../notebooks/_bootstrap.py

In [ ]:
# ===== 共享规格 (单一数据源, 与 06 汇总一致) =====
from examples._examples_spec import (
    EXAMPLES, run_example, phase_files, compare_xemit)
NAME = "Manual_Example"
STEM = "Example"
print("算例:", NAME, "| 物理:", EXAMPLES[NAME]["title"])


In [ ]:
# ===== 运行本算例 (generator/astra 按需) =====
work = run_example(NAME)
print("输出文件:")
for f in sorted(work.glob(STEM + ".*")):
    print("  ", f.name)

In [ ]:
# ===== 束团统计 (最后一个 z 位置) =====
from astra_tools.io import read_distribution
from astra_tools.analysis.statistics import compute_statistics, print_statistics
from astra_tools.widgets.panels import stats_table_html
ph = phase_files(work, STEM)
dist = read_distribution(ph[-1])
print("相空间文件:", ph[-1].name)
print_statistics(compute_statistics(dist),
                 title="%s @ %s" % (NAME, ph[-1].name))
stats_table_html(compute_statistics(dist))

### 教学要点

这是手册的主算例: 光阴极枪产生束团, 螺线管做发射度补偿, 追踪到 1.5 m。它生成了全套输出 (Xemit/Zemit/Sigma/相空间), 因此用最全的图组展示。

In [ ]:
from astra_tools.plot.phase_space import plot_transverse_phase_space, plot_phase_space
# 1 GeV 束流发散角 ~1 urad: 原始图是贴地横线 (物理正确), normalize 后结构可见
plot_transverse_phase_space(dist)
plot_phase_space(dist, plane="x", normalize=True, title="x-x 归一化")
plot_phase_space(dist, plane="z")

In [ ]:
from astra_tools.plot.overview import plot_overview
plot_overview(dist)

In [ ]:
from astra_tools.analysis.slices import compute_slice_analysis
from astra_tools.plot.slice_plots import plot_slice_dashboard
from astra_tools.plot.advanced_plots import plot_slice_mismatch
sa = compute_slice_analysis(dist, n_slices=20)
plot_slice_dashboard(sa)
plot_slice_mismatch(dist, n_slices=20)

In [ ]:
from astra_tools.io.astra_emit import read_emit_files, read_ref_file
from astra_tools.plot.emit_plots import (
    plot_emit_dashboard, plot_envelope_evolution,
    plot_velocity_evolution, plot_step_size_evolution)
emit = read_emit_files(str(work / STEM))
ref = read_ref_file(str(work / STEM))
plot_emit_dashboard(emit)
plot_envelope_evolution(emit, x_axis="t")
plot_velocity_evolution(ref)
plot_step_size_evolution(ref)

In [ ]:
from astra_tools.plot.advanced_plots import plot_beta_alpha, plot_phase_advance
plot_beta_alpha(emit)
plot_phase_advance(emit)

In [ ]:
from astra_tools.io.astra_emit import read_sigma_file
from astra_tools.plot.emit_plots import plot_eigen_emittances
sigma = read_sigma_file(str(work / STEM))
plot_eigen_emittances(sigma)  # 动量列/mc 之谜已破解, 与 Xemit 对照 <8%

In [ ]:
from astra_tools.export import export_distribution, export_statistics
out = work / "export"
print("导出目录:", out)
print(export_distribution(dist, out))
print(export_statistics(compute_statistics(dist), out))

In [ ]:
# ===== 黄金比对 (rel < 0.5% 判 OK) =====
compare_xemit(NAME, work)
print("返回汇总: notebooks/06_examples.ipynb")